In [1]:
# pip install datasets

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
from datasets import load_dataset
import matplotlib.pyplot as plt

## Config

In [3]:
model_name = "gpt2"
dataset_name = "tatsu-lab/alpaca"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

batch_size = 4
ppo_epochs = 3
learning_rate = 1e-5
cliprange_ratio = 0.2
max_length = 32
gen_length = 5  # generate 5 tokens per prompt for reward

## Load Dataset

In [4]:
dataset = load_dataset(dataset_name)

## Settings

In [ ]:
def combine_example(ex):
    if ex['input'].strip() == "":
        return ex['instruction'] + "\nAnswer:"
    else:
        return ex['instruction'] + "\n" + ex['input'] + "\nAnswer:"

dataset = dataset.map(lambda x: {'text': combine_example(x)}, remove_columns=dataset['train'].column_names)
split_data = dataset['train'].train_test_split(test_size=0.2)
train_texts = split_data['train']['text']
val_texts = split_data['test']['text']

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=max_length):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True, padding='max_length', max_length=self.max_len, return_tensors='pt')
        return {k: v.squeeze(0) for k, v in enc.items()}

train_dataset = TextDataset(train_texts, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

/venv/main/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


## Reward Model

In [6]:
class GPT2RewardModel(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.transformer = AutoModel.from_pretrained(model_name)
        hidden_size = self.transformer.config.hidden_size
        self.reward_head = nn.Linear(hidden_size, 1)
    def forward(self, input_ids, attention_mask=None):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state
        last_token = last_hidden[:, -1, :]
        reward = self.reward_head(last_token)
        return reward.squeeze(-1)

reward_model = GPT2RewardModel(model_name).to(device)
reward_model.eval()

GPT2RewardModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D()
          (c_proj): Conv1D()
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (reward_head): Linear(in_features=768, out_features=1, bias=True)
)

## Policy Model

In [7]:
policy_model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
old_policy_model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
old_policy_model.load_state_dict(policy_model.state_dict())
old_policy_model.eval()

optimizer = optim.Adam(policy_model.parameters(), lr=learning_rate)

## PPO Training Loop

In [ ]:
train_losses = []

for epoch in range(ppo_epochs):
    policy_model.train()
    epoch_loss = 0

    for i, batch in enumerate(train_loader, 1):
        batch = {k: v.to(device) for k, v in batch.items()}

        # --- 1️⃣ Generate tokens from policy ---
        outputs = policy_model.generate(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            max_length=batch["input_ids"].shape[1] + gen_length,
            do_sample=True,
            top_k=50
        )

        # Only consider the generated tokens
        gen_ids = outputs[:, batch["input_ids"].shape[1]:]

        # --- 2️⃣ Compute reward for new policy ---
        with torch.no_grad():
            rewards = reward_model(torch.cat([batch["input_ids"], gen_ids], dim=1))

        # --- 3️⃣ Compute reward for old policy ---
        with torch.no_grad():
            old_outputs = old_policy_model.generate(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                max_length=batch["input_ids"].shape[1] + gen_length,
                do_sample=True,
                top_k=50
            )
            old_gen_ids = old_outputs[:, batch["input_ids"].shape[1]:]
            old_rewards = reward_model(torch.cat([batch["input_ids"], old_gen_ids], dim=1))

        # --- 4️⃣ Compute advantage ---
        advantage = rewards - old_rewards

        # --- 5️⃣ Compute log-probs of generated tokens ---
        logits = policy_model(input_ids=torch.cat([batch["input_ids"], gen_ids], dim=1)).logits
        log_probs = torch.log_softmax(logits, dim=-1)
        # Gather log-probs of the generated tokens
        generated_log_probs = log_probs[:, -gen_length:, :]
        selected_log_probs = generated_log_probs.gather(-1, gen_ids.unsqueeze(-1)).squeeze(-1)
        selected_log_probs = selected_log_probs.mean(dim=1)  # mean over generated tokens

        # --- 6️⃣ Old log-probs ---
        with torch.no_grad():
            old_logits = old_policy_model(input_ids=torch.cat([batch["input_ids"], old_gen_ids], dim=1)).logits
            old_log_probs_full = torch.log_softmax(old_logits, dim=-1)
            old_selected_log_probs = old_log_probs_full[:, -gen_length:, :].gather(-1, old_gen_ids.unsqueeze(-1)).squeeze(-1)
            old_selected_log_probs = old_selected_log_probs.mean(dim=1)

        # --- 7️⃣ PPO Surrogate Loss ---
        ratio = torch.exp(selected_log_probs - old_selected_log_probs)
        surrogate1 = ratio * advantage
        surrogate2 = torch.clamp(ratio, 1 - cliprange_ratio, 1 + cliprange_ratio) * advantage
        loss = -torch.mean(torch.min(surrogate1, surrogate2))

        # --- 8️⃣ Backprop ---
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        print("loss", loss.item())
        epoch_loss += loss.item()
        if i % 100 == 0 or i == len(train_loader):
            print(f"Epoch {epoch+1}, Batch {i}, Loss: {loss.item():.4f}")

    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{ppo_epochs}, Loss: {avg_loss:.4f}")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


loss 1.4292967319488525
loss 0.24875257909297943


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


loss -0.23231011629104614
loss 0.2773536741733551


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


...

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


loss 0.48686885833740234
loss 0.38986724615097046


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


loss 0.13070151209831238
loss 0.6984127759933472


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


## Plot Loss

In [ ]:
plt.plot(train_losses, label="Train Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("PPO Training Loss")
plt.legend()
plt.show()

In [ ]:
policy_model.eval()  # set to evaluation mode
reward_model.eval()

def evaluate_policy(prompts, tokenizer, policy_model, reward_model, max_len=32, gen_len=5):
    """
    Generates responses for prompts, computes rewards, and prints them.
    """
    results = []
    for prompt in prompts:
        # Tokenize prompt
        enc = tokenizer(prompt, return_tensors="pt").to(device)
        
        # Generate sequence from trained policy
        with torch.no_grad():
            output_ids = policy_model.generate(
                input_ids=enc["input_ids"],
                attention_mask=enc["attention_mask"],
                max_length=enc["input_ids"].shape[1]+gen_len,
                do_sample=True,
                top_k=50
            )
            
            # Only generated tokens
            gen_ids = output_ids[:, enc["input_ids"].shape[1]:]
            
            # Compute reward for generated sequence
            reward = reward_model(torch.cat([enc["input_ids"], gen_ids], dim=1))
        
        # Decode generated text
        generated_text = tokenizer.decode(gen_ids[0], skip_special_tokens=True)
        results.append((prompt, generated_text, reward.item()))
    
    return results

## Example evaluation

In [ ]:
val_prompts = [
    "Explain what machine learning is.",
    "Summarize the following: AI models require data and computation."
]

eval_results = evaluate_policy(val_prompts, tokenizer, policy_model, reward_model)

for i, (prompt, response, reward) in enumerate(eval_results):
    print(f"Prompt {i+1}: {prompt}")
    print(f"Generated: {response}")
    print(f"Reward: {reward:.4f}\n")